# run.ipynb — one-click pipeline for THIS pod bundle

Runs, in order and each in its **own fresh kernel**:

`setup.ipynb` → `prepare_training.ipynb` → `training.ipynb`

**Fail-fast**: if any cell in a child notebook errors, `nbclient` raises
`CellExecutionError` here and the chain stops — later notebooks never start,
exactly like an uncaught exception in a plain Python script. The partially
executed copy (including the traceback) is saved next to this file as
`executed_<name>.ipynb`, so you can open it and see the failing cell.

Notes:
- Child notebooks are read from **this bundle's folder**, so the per-VM
  `VM_NAME` baked in by `prepare_pods.ipynb` is used as-is. Nothing to edit here.
- A child cell's output appears below **after that cell finishes**. The
  multi-hour training cell is therefore silent here while it runs — follow it
  live via `realtime_reader.ipynb` / the per-VM log, as usual.
- Re-running is safe: setup/prepare skip finished work, training resumes and
  re-claims combos through the shared coordinator.

In [ ]:
# nbclient/nbformat ship with JupyterLab on the pods -- this is a no-op there.
try:
    import nbclient, nbformat  # noqa: F401
except ImportError:
    %pip install -q nbclient nbformat
    import nbclient, nbformat  # noqa: F401
print('nbclient', nbclient.__version__, '| nbformat', nbformat.__version__)

In [ ]:
# Execute the pipeline. Stops at the FIRST error (CellExecutionError propagates).
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

HERE = Path.cwd()
NOTEBOOKS = ['setup.ipynb', 'prepare_training.ipynb', 'training.ipynb']


def echo_cell_output(cell=None, cell_index=None, execute_reply=None, **kwargs):
    # Relay each finished cell's stream output into THIS notebook, so run.ipynb
    # reads like one continuous log of the whole pipeline.
    for out in (cell or {}).get('outputs', []):
        if out.get('output_type') == 'stream':
            print(out.get('text', ''), end='', flush=True)


for name in NOTEBOOKS:
    out_path = HERE / f'executed_{name}'
    print(f"\n{'=' * 70}\n==  {name}  (started {time.strftime('%Y-%m-%d %H:%M:%S')})\n{'=' * 70}",
          flush=True)
    nb = nbformat.read(HERE / name, as_version=4)
    client = NotebookClient(
        nb,
        timeout=None,                      # the training cell legitimately runs for hours
        kernel_name='python3',
        resources={'metadata': {'path': str(HERE)}},
        on_cell_executed=echo_cell_output,
    )
    try:
        client.execute()
    except CellExecutionError:
        print(f'\n!! {name} FAILED -- chain aborted; later notebooks were NOT run.\n'
              f'!! Open {out_path.name} to see the failing cell + traceback.', flush=True)
        raise
    finally:
        nbformat.write(nb, out_path)       # keep outputs either way (success or failure)
    print(f'==  {name} OK -> {out_path.name}', flush=True)

print('\nALL NOTEBOOKS DONE. Next: check_paralle.ipynb (coordination view), '
      'realtime_reader.ipynb (live log), eval.ipynb (after the whole sweep).')